# 04 — Imputation Analysis
**NANO Study | ESD Lab, University of South Carolina**

> ⚠️ **HIPAA Notice**: Synthetic data only. Real imputation uses `src/imputation/mice_imputation.R` via R subprocess or `rpy2` bridge.

## Goals
1. Demonstrate MICE workflow via Python (sklearn IterativeImputer) on synthetic longitudinal data
2. Assess imputation convergence diagnostics
3. Compare imputed vs observed distributions
4. Apply Rubin's rules to pool estimates across imputed datasets

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from sklearn.experimental import enable_iterative_imputer  # noqa
    from sklearn.impute import IterativeImputer
    SKLEARN_AVAILABLE = True
    print('sklearn IterativeImputer available.')
except ImportError:
    SKLEARN_AVAILABLE = False
    print('sklearn not available — using simpler mean imputation demo.')

rng = np.random.default_rng(seed=42)
print('Setup complete.')

## 1. Generate Synthetic Longitudinal Dataset with MAR Missingness

In [ ]:
N = 120  # participants
T = 5    # timepoints (months: 1, 3, 6, 12, 36)
timepoints = [1, 3, 6, 12, 36]
GROUPS = ['ASIB', 'PT', 'TD']

rows = []
for i in range(N):
    group = rng.choice(GROUPS, p=[0.2, 0.5, 0.3])
    ga = rng.normal(29, 2)
    sex = rng.choice([0, 1])
    ses = rng.normal(45, 10)
    intercept = rng.normal(4.2, 0.8) + (0.3 if group == 'ASIB' else 0)
    slope = rng.normal(-0.05, 0.02)
    for j, tp in enumerate(timepoints):
        rsa = intercept + slope * tp + rng.normal(0, 0.3)
        rmssd = rng.normal(35 - 0.3 * j, 8)
        ados = rng.normal(8, 3) if group == 'ASIB' and tp == 36 else np.nan
        miss_prob = 0.04 * j + (0.08 if group == 'ASIB' else 0)
        rows.append({
            'id': f'NANO-{i+1:04d}',
            'group': group,
            'timepoint': tp,
            'ga_weeks': round(ga, 1),
            'sex': sex,
            'ses_score': round(ses, 1),
            'rsa': np.nan if rng.random() < miss_prob else round(rsa, 3),
            'rmssd_ms': np.nan if rng.random() < miss_prob + 0.05 else round(rmssd, 2),
            'ados_css': np.nan if np.isnan(ados) else max(1, round(ados)),
        })

df = pd.DataFrame(rows)
print(f'Dataset: {df.shape} | Missing RSA: {df["rsa"].isna().sum()} ({100*df["rsa"].isna().mean():.1f}%)')
print(f'Missing RMSSD: {df["rmssd_ms"].isna().sum()} ({100*df["rmssd_ms"].isna().mean():.1f}%)')
df.head(10)

## 2. Observed vs Imputed Distribution Comparison

In [ ]:
feature_cols = ['ga_weeks', 'ses_score', 'rsa', 'rmssd_ms']
X = df[feature_cols].values

if SKLEARN_AVAILABLE:
    imputer = IterativeImputer(max_iter=10, random_state=42, verbose=0)
    X_imputed = imputer.fit_transform(X)
    df_imputed = df.copy()
    df_imputed[feature_cols] = X_imputed
else:
    df_imputed = df.copy()
    for col in feature_cols:
        df_imputed[col] = df_imputed[col].fillna(df_imputed[col].mean())

# Compare distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['rsa', 'rmssd_ms']):
    observed = df[col].dropna()
    was_missing = df[col].isna()
    imputed_vals = df_imputed.loc[was_missing, col]
    ax.hist(observed, bins=25, alpha=0.6, label='Observed', color='steelblue', density=True)
    ax.hist(imputed_vals, bins=15, alpha=0.6, label='Imputed', color='tomato', density=True)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.set_title(f'{col} — Observed vs Imputed')
    ax.legend()
plt.suptitle('Distribution Check: Imputed Values vs Observed Values', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Multiple Imputation & Rubin's Rules

We generate M=20 imputed datasets, fit a linear model on each, then pool estimates using Rubin's (1987) rules.

In [ ]:
from scipy import stats as sp_stats

M = 20
estimates = []
variances = []

for m in range(M):
    if SKLEARN_AVAILABLE:
        imp = IterativeImputer(max_iter=5, random_state=m)
        X_m = imp.fit_transform(X)
        df_m = pd.DataFrame(X_m, columns=feature_cols)
    else:
        df_m = df[feature_cols].copy()
        for col in feature_cols:
            df_m[col] = df_m[col].fillna(df_m[col].mean() + rng.normal(0, 0.1 * df_m[col].std()))

    # Simple linear regression: rsa ~ ga_weeks + ses_score
    y_m = df_m['rsa'].values
    X_m_reg = np.column_stack([np.ones(len(df_m)), df_m['ga_weeks'].values,
                                df_m['ses_score'].values])
    try:
        result = np.linalg.lstsq(X_m_reg, y_m, rcond=None)
        beta = result[0]
        residuals = y_m - X_m_reg @ beta
        sigma2 = np.var(residuals, ddof=X_m_reg.shape[1])
        XtX_inv = np.linalg.inv(X_m_reg.T @ X_m_reg)
        var_beta = sigma2 * np.diag(XtX_inv)
        estimates.append(beta)
        variances.append(var_beta)
    except np.linalg.LinAlgError:
        continue

estimates = np.array(estimates)
variances = np.array(variances)

# Rubin's rules pooling
Q_bar = estimates.mean(axis=0)          # pooled estimate
W = variances.mean(axis=0)              # within-imputation variance
B = np.var(estimates, axis=0, ddof=1)   # between-imputation variance
T_var = W + (1 + 1/M) * B              # total variance
se = np.sqrt(T_var)

param_names = ['Intercept', 'GA weeks', 'SES score']
results_df = pd.DataFrame({
    'Parameter': param_names,
    'Estimate (Q̄)': Q_bar.round(4),
    'SE (pooled)': se.round(4),
    't': (Q_bar / se).round(3),
    'p (approx)': [round(2 * (1 - sp_stats.t.cdf(abs(t), df=M-1)), 4)
                   for t in Q_bar / se],
    '95% CI lower': (Q_bar - 1.96 * se).round(4),
    '95% CI upper': (Q_bar + 1.96 * se).round(4),
    'FMI (frac. missing)': ((B / T_var) * (1 + 1/M)).round(3),
})
print(f'Pooled results across M={M} imputed datasets (Rubin\'s Rules):')
display(results_df)

## Summary
- IterativeImputer (sklearn) approximates MICE for Python-native workflow
- Imputed distributions are broadly consistent with observed distributions
- Rubin's rules appropriately inflate SEs to account for imputation uncertainty
- Fraction of missing information (FMI) quantifies impact of missingness on each parameter

**Production**: Full multi-level MICE (clustered longitudinal) is run in R via `src/imputation/mice_imputation.R`

**Next**: See `05_ml_model_development.ipynb` for full ML pipeline.